# EDA & Cleaning: MIMIC-III Clinical Database Demo

Source: `mimic3_demo/mimic-iii-clinical-database-demo-1.4/`. Real (but small, de-identified demo) ICU patients. No genotype data, but has warfarin prescriptions, INR lab trajectories, admissions, and diagnosis history. Useful as a **clinical-history feature source** and a small real-world validation set for the non-genetic part of the model.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", 100)
DATA_DIR = Path("..").resolve()
DECISIONS_PATH = Path("dataset_decisions.json")

def record_decision(name, verdict, reason, n_rows=None, n_cols=None, saved_to=None):
    decisions = {}
    if DECISIONS_PATH.exists():
        decisions = json.loads(DECISIONS_PATH.read_text())
    decisions[name] = {
        "verdict": verdict,
        "reason": reason,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "saved_to": saved_to,
    }
    DECISIONS_PATH.write_text(json.dumps(decisions, indent=2))
    print(f"Recorded decision for '{name}': {verdict}")


## File inventory: `mimic3_demo/mimic-iii-clinical-database-demo-1.4/` (28 data tables plus license/checksum files)

**Used:**
- `warfarin_prescriptions.xlsx`, `warfarin_patients_inr_trajectory.xlsx`: already-filtered extracts (not raw MIMIC-III tables) built from `PRESCRIPTIONS.xlsx` and `inr_labevents.xlsx`/`LABEVENTS.xlsx`, restricted to warfarin orders and INR/PT results.
- `ADMISSIONS.xlsx`: admission-level history features (admission type, diagnosis text, etc).
- `PATIENTS.xlsx`: age and gender, merged into the cleaned admissions table below.
- `DIAGNOSES_ICD.xlsx`: diagnosis history (ICD-9 codes).

**Not used:** `CALLOUT`, `CAREGIVERS`, `CHARTEVENTS` (bulk vitals/monitoring), `CPTEVENTS`/`D_CPT`/`DRGCODES` (billing codes), `D_ICD_DIAGNOSES`/`D_ICD_PROCEDURES`/`D_ITEMS`/`D_LABITEMS` (code dictionaries, not joined for human-readable labels), `DATETIMEEVENTS`, `ICUSTAYS`, `INPUTEVENTS_CV`/`INPUTEVENTS_MV` (fluid/medication infusions), `inr_labevents.xlsx`/`LABEVENTS.xlsx` (superseded by the pre-filtered trajectory file), `MICROBIOLOGYEVENTS`, `NOTEEVENTS` (free text), `OUTPUTEVENTS`, `PROCEDUREEVENTS_MV`, `PROCEDURES_ICD`, `SERVICES`, `TRANSFERS`. None of these carry warfarin, INR, demographic, or diagnosis signal beyond what is already extracted.

In [ ]:
base = DATA_DIR / "usable/mimic3_demo/mimic-iii-clinical-database-demo-1.4"
rx = pd.read_excel(base / "warfarin_prescriptions.xlsx")
inr_traj = pd.read_excel(base / "warfarin_patients_inr_trajectory.xlsx")
admissions = pd.read_excel(base / "ADMISSIONS.xlsx")
patients = pd.read_excel(base / "PATIENTS.xlsx")
diagnoses = pd.read_excel(base / "DIAGNOSES_ICD.xlsx")
print("warfarin_prescriptions:", rx.shape)
print("inr_trajectory:", inr_traj.shape)
print("admissions:", admissions.shape)
print("patients:", patients.shape)
print("diagnoses:", diagnoses.shape)

In [ ]:
n_warfarin_patients = rx["subject_id"].nunique()
print(f"Distinct patients with a warfarin prescription: {n_warfarin_patients}")
rx.head(3)

In [ ]:
inr_traj.head(3)

## Missingness in the derived tables

In [ ]:
for name, d in [("prescriptions", rx), ("inr_trajectory", inr_traj)]:
    print(f"--- {name} ---")
    print((d.isna().mean() * 100).round(1))

## Cleaning
Keep only rows for patients who actually appear in the warfarin cohort. Drop the very large raw event tables (CHARTEVENTS, NOTEEVENTS, LABEVENTS) from the modeling pipeline. They are needed only to derive the trajectory/prescription tables already extracted here, not as direct model inputs.

In [ ]:
warfarin_ids = rx["subject_id"].unique()
adm_clean = admissions[admissions["subject_id"].isin(warfarin_ids)].copy()
dx_clean = diagnoses[diagnoses["subject_id"].isin(warfarin_ids)]

# Merge in age (at admission) and gender from PATIENTS.xlsx.
adm_clean = adm_clean.merge(patients[["subject_id", "gender", "dob"]], on="subject_id", how="left")
adm_clean["admittime"] = pd.to_datetime(adm_clean["admittime"])
adm_clean["dob"] = pd.to_datetime(adm_clean["dob"])
adm_clean["age_years"] = (adm_clean["admittime"] - adm_clean["dob"]).dt.days / 365.25
# MIMIC-III de-identifies ages over 89 by shifting DOB into the far past. Cap at 89 per MIMIC convention.
adm_clean["age_years"] = adm_clean["age_years"].clip(upper=89).round(1)
adm_clean = adm_clean.drop(columns=["dob"])

print("admissions for warfarin cohort:", adm_clean.shape)
print("diagnoses for warfarin cohort:", dx_clean.shape)
print("age_years summary:\n", adm_clean["age_years"].describe())
rx.to_csv("cleaned/mimic3_warfarin_prescriptions_clean.csv", index=False)
inr_traj.to_csv("cleaned/mimic3_inr_trajectory_clean.csv", index=False)
adm_clean.to_csv("cleaned/mimic3_admissions_clean.csv", index=False)
dx_clean.to_csv("cleaned/mimic3_diagnoses_clean.csv", index=False)
print("Saved 4 cleaned tables to cleaned/")

## Verdict

**KEEP: real-world clinical-history validation source (no genotype).** Small cohort (demo release), but genuinely real patients with longitudinal INR and admission/diagnosis history. Useful for the clinical-only arm of the model and for testing generalization beyond IWPC. Raw bulk event tables excluded from the modeling pipeline.

In [ ]:
record_decision(
    "mimic3_demo",
    verdict="KEEP",
    reason="Real (demo-scale) patients with warfarin Rx + INR trajectory + admission/diagnosis history; no genotype. Use for clinical-only validation.",
    n_rows=int(n_warfarin_patients), n_cols=None, saved_to="cleaned/mimic3_*_clean.csv",
)